# Research graph: inexpensive extraction + Jev
Question: which inexpensive extractor preserves page-grounded entities and directed relationships in French BRH Markdown?
Decision rule fixed before the run: 30 human-reviewed pages across prose, tables and regulations; entity recall ≥ 80%, accepted relationship precision ≥ 95% in each family, and 100% structural evidence validity. Compare three seeds (17, 41, 97), report spread, alias mistakes and observed cost. Choose the cheapest eligible configuration only after human sign-off.
Offline by default. Outputs are recommendations; no product defaults or production graph publication follow automatically. Community/global GraphRAG is outside this experiment.

In [ ]:
from pathlib import Path
import json, subprocess
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'packages/core').is_dir())
OUT = ROOT / 'experiments/research-graph'
manifest = json.loads((OUT / 'manifest.json').read_text())
print(manifest['decision_rule'])
print('Pages:', len(manifest['pages']), 'Reviewed:', sum(bool(p['reviewed_by']) for p in manifest['pages']))

## Review before quality scoring
Open each referenced `data/ocr/<sha>/document.md` at its physical page marker. Fill `expected_entities` with `kind`, `label`, and reviewed aliases; fill `expected_relationships` with `subject`, `predicate`, `object`. Record the human reviewer in `reviewed_by`. Include negatives and ambiguous acronyms. Do not edit labels to match predictions. Preserve artifact hashes.
The driver is notebook glue and imports shipped adapters for protocol consistency; shipped code never imports this directory.

In [ ]:
ALLOW_PAID = False
if ALLOW_PAID:
    subprocess.run(['uv', 'run', 'python', 'experiments/research-graph/run.py', 'run', '--allow-paid'], cwd=ROOT, check=True)
else:
    print('Skipped: paid model calls require ALLOW_PAID=True and the existing OpenRouter key.')

In [ ]:
subprocess.run(['uv', 'run', 'python', 'experiments/research-graph/run.py', 'score'], cwd=ROOT, check=True)
scores = json.loads((OUT / 'scores.json').read_text())
print((OUT / 'REPORT.md').read_text())
scores

## Answer and trade-off
Pending human-reviewed labels and comparative runs. No quality winner is declared. The committed protocol smoke (when present) only checks provider access, schema, quoted evidence and Jev response parsing. A cheap LLM plus a classifier can still miss entities, misresolve aliases or accept unsupported facts; cost alone does not select production behaviour.